<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-07b-entity-resolution-with-weak-labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 7 (continued) - Who is this? Matching noisy records without labels: blocking, similarity, weak labels and clusters

Two systems describe the same person (or the same browser, or the same device) in slightly different, noisy ways, and nobody has labelled which records go together. This lab builds the standard pipeline - **block, compare, label weakly, learn, group, audit** - on a public benchmark where the truth happens to be known, so you can see how far the no-label methods get and where they fail.

**Optional advanced-discussion lab** for Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Runs on the free Colab CPU in about a minute.

*Original code written for this course. Data: the Febrl4 benchmark records that ship with the open-source `recordlinkage` package (BSD-3). The records were produced by the Febrl data generator, which copies typical real-world errors (typos, swapped fields, missing values), so treat it as a realistic exercise, not as real people. Technique names follow the entity-resolution literature (Fellegi and Sunter 1969; blocking; programmatic weak supervision as in the Snorkel papers, Ratner et al.).*

In [ ]:
%pip install -q recordlinkage rapidfuzz scikit-learn scipy pandas numpy

In [ ]:
import numpy as np, pandas as pd, itertools, time
from recordlinkage.datasets import load_febrl4
from rapidfuzz.distance import JaroWinkler, Levenshtein
np.random.seed(0)

A, B, true_links = load_febrl4(return_links=True)       # two files of 5,000 records; 5,000 records in A have a partner in B
print("file A:", A.shape, "| file B:", B.shape, "| records that truly match:", len(true_links))
print("pairs to consider if we compare everything:", f"{len(A)*len(B):,}")
A.head(4)

## Part 1 - Blocking: do not compare everything

Comparing every record in A with every record in B is 25 million pairs here and impossible at the scale of a real identification platform. **Blocking** keeps only pairs that agree on a cheap key. Because every key can be corrupted by a typo, we use several keys and take the *union*. Two numbers judge a blocking scheme:

* **Reduction ratio** - what fraction of the 25 million pairs we avoid.
* **Pair completeness** - what fraction of the true matches survive into the candidate set (only measurable here because we hold the truth; on real data you estimate it from a small labelled sample).

In [ ]:
def block_pairs(A, B, keys):
    """Candidate pairs (index of A, index of B) sharing the value of at least one blocking key."""
    out = set()
    for name, f in keys.items():
        a = A.assign(_k=f(A))[lambda d: d._k.notna() & (d._k != "")]["_k"].reset_index().rename(columns={"rec_id": "ia"})
        b = B.assign(_k=f(B))[lambda d: d._k.notna() & (d._k != "")]["_k"].reset_index().rename(columns={"rec_id": "ib"})
        m = a.merge(b, on="_k")
        out |= set(zip(m.ia, m.ib))
    return out

truth = set(true_links)                       # (id in A, id in B) pairs
total = len(A) * len(B)
schemes = {
    "surname only":                      {"surname": lambda d: d.surname},
    "surname + postcode":                {"surname": lambda d: d.surname, "postcode": lambda d: d.postcode},
    "surname + postcode + birth date":   {"surname": lambda d: d.surname, "postcode": lambda d: d.postcode, "dob": lambda d: d.date_of_birth},
    "above + given name + suburb":       {"surname": lambda d: d.surname, "postcode": lambda d: d.postcode, "dob": lambda d: d.date_of_birth,
                                          "given": lambda d: d.given_name, "suburb": lambda d: d.suburb},
}
print(f"{'blocking keys (union)':36}{'pairs':>10}{'reduction':>11}{'completeness':>14}")
cands = {}
for name, keys in schemes.items():
    c = block_pairs(A, B, keys); cands[name] = c
    print(f"{name:36}{len(c):>10,}{1-len(c)/total:>11.4%}{len(c & truth)/len(truth):>14.1%}")

**Read it.** Each extra key buys completeness and costs pairs; the trade-off is the whole game. We take the last scheme: it keeps almost every true match while still discarding more than 99.9% of the 25 million pairs. The matches that blocking loses can *never* be recovered later, so completeness is a ceiling on everything downstream.

## Part 2 - Compare: turn each candidate pair into numbers

**We deliberately withhold the `soc_sec_id` field.** A clean unique identifier is a luxury - browsers and devices rarely have one, and if you had it you would simply join on it - so we make the problem as hard as the real one. For each pair we compute a similarity per field: Jaro-Winkler for names (forgiving of typos and strong on shared prefixes), normalised edit similarity for identifiers and addresses, exact agreement for dates and codes. A missing value is its own state, not a mismatch.

In [ ]:
cand = sorted(cands["above + given name + suburb"])
ia, ib = zip(*cand)
RA, RB = A.loc[list(ia)].reset_index(), B.loc[list(ib)].reset_index()

def sim(col, kind):
    a, b = RA[col].fillna("").astype(str).values, RB[col].fillna("").astype(str).values
    if kind == "jw":  f = lambda x, y: JaroWinkler.normalized_similarity(x, y)
    elif kind == "lev": f = lambda x, y: Levenshtein.normalized_similarity(x, y)
    else:             f = lambda x, y: 1.0 if x == y else 0.0
    return np.array([np.nan if (x == "" or y == "") else f(x, y) for x, y in zip(a, b)])

t0 = time.time()
X = pd.DataFrame({
    "given_jw":   sim("given_name", "jw"),   "surname_jw": sim("surname", "jw"),
    "street_no":  sim("street_number", "eq"), "addr1_lev":  sim("address_1", "lev"),
    "addr2_lev":  sim("address_2", "lev"),    "suburb_jw":  sim("suburb", "jw"),
    "postcode_eq": sim("postcode", "eq"),     "state_eq":   sim("state", "eq"),
    "dob_eq":     sim("date_of_birth", "eq"),
})
y_true = np.array([(a, b) in truth for a, b in zip(RA.rec_id, RB.rec_id)])      # used ONLY to grade, never to train
print(f"{len(X):,} candidate pairs, {y_true.sum():,} true matches ({y_true.mean():.2%}), features built in {time.time()-t0:.1f}s")
print("share missing per feature:\n", X.isna().mean().round(3).to_string())

## Part 3 - Weak labels: write rules instead of labelling pairs

Nobody has time to label 100,000 pairs. **Programmatic weak supervision** writes a handful of cheap, imperfect rules - *labelling functions* (LFs) - each of which votes **match (1)**, **non-match (0)** or **abstain (-1)**. We then combine the votes. Rules are allowed to be wrong; they are not allowed to be systematically wrong in the same direction, so we write some that look for agreement and some that look for contradiction.

In [ ]:
ABS = -1
def lf_street(r):    return 1 if (r.street_no == 1 and r.addr1_lev >= 0.8 and r.surname_jw >= 0.8) else ABS   # same house number, similar street, similar surname
def lf_dob_name(r):  return 1 if (r.dob_eq == 1 and r.surname_jw >= 0.9) else ABS                 # same birth date and a close surname
def lf_names(r):     return 1 if (r.given_jw >= 0.92 and r.surname_jw >= 0.92) else (0 if (r.given_jw < 0.6 and r.surname_jw < 0.6) else ABS)
def lf_address(r):   return 1 if (r.addr1_lev >= 0.9 and r.postcode_eq == 1 and r.suburb_jw >= 0.9) else ABS
def lf_state(r):     return 0 if (r.state_eq == 0 and r.postcode_eq == 0) else ABS               # different state AND postcode: unlikely the same person
def lf_dob_diff(r):  return 0 if (r.dob_eq == 0 and r.surname_jw < 0.75 and r.given_jw < 0.75) else ABS

LFS = [lf_street, lf_dob_name, lf_names, lf_address, lf_state, lf_dob_diff]
Xf = X.fillna({c: 0.5 for c in X.columns})                       # neutral value for missing similarity, so rules abstain rather than crash
L = np.array([[lf(r) for lf in LFS] for r in Xf.itertuples(index=False)])

print(f"{'labelling function':16}{'coverage':>10}{'votes match':>13}{'votes non-match':>17}   (no truth used yet)")
for j, lf in enumerate(LFS):
    v = L[:, j]; print(f"{lf.__name__:16}{(v!=ABS).mean():>10.1%}{(v==1).mean():>13.1%}{(v==0).mean():>17.1%}")
print("pairs where rules disagree with each other:", f"{(((L==1).any(1)) & ((L==0).any(1))).mean():.2%}", "| pairs no rule votes on:", f"{(L==ABS).all(1).mean():.2%}")

**Combine the votes.** The simplest rule is a majority vote among the rules that did not abstain, with ties and all-abstain pairs left *unlabelled*. We keep the unlabelled pairs out of training rather than guessing. Only now do we peek at the truth, once, to grade the weak labels.

In [ ]:
def majority(L):
    ones, zeros = (L == 1).sum(1), (L == 0).sum(1)
    out = np.full(len(L), ABS); out[ones > zeros] = 1; out[zeros > ones] = 0
    return out

weak = majority(L); lab = weak != ABS
def prf(pred, truth_):
    tp = ((pred == 1) & truth_).sum(); fp = ((pred == 1) & ~truth_).sum(); fn = ((pred != 1) & truth_).sum()
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1); return p, r, 2 * p * r / max(p + r, 1e-9)
p, r, f = prf(weak[lab], y_true[lab])
print(f"weakly labelled: {lab.mean():.1%} of pairs | on those pairs precision {p:.3f}, recall {r:.3f}, F1 {f:.3f}")
print(f"matches the rules never labelled at all: {int(((~lab) & y_true).sum())} of {int(y_true.sum())}")

## Part 4 - Learn from the weak labels

A model trained on the weak labels is useful for two reasons: it is **smoother** than any single rule (it combines all nine similarity features, not only the ones the rules looked at), and it **scores every pair**, including the ones the rules abstained on. We use gradient boosting, which accepts missing values natively and is the usual workhorse for tabular pair features.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
rng = np.random.default_rng(0)
order = rng.permutation(len(X)); split = len(X) // 2
tr, te = order[:split], order[split:]                      # split the candidate pairs; grade only on the held-out half

def fit_eval(train_idx, labels, name):
    m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=0).fit(X.iloc[train_idx], labels)
    proba = m.predict_proba(X.iloc[te])[:, 1]
    p, r, f = prf((proba >= 0.5).astype(int), y_true[te])
    print(f"{name:56}precision {p:.3f}  recall {r:.3f}  F1 {f:.3f}")
    return m, proba

tr_lab = tr[lab[tr]]
fit_eval(tr_lab, weak[tr_lab], "A: weak labels, abstained pairs left out of training")
weak0 = np.where(weak == ABS, 0, weak)                    # B: matches are rare (about 2%), so an unlabelled pair is far more likely a non-match
m_weak, p_weak = fit_eval(tr, weak0[tr], "B: weak labels, abstained pairs treated as non-match")
small = tr[rng.permutation(len(tr))[:200]]
fit_eval(small, y_true[small], "C: 200 human-labelled pairs only")
mix = np.concatenate([tr, small]); mix_y = np.concatenate([weak0[tr], y_true[small]])
m_mix, p_mix = fit_eval(mix, mix_y, "D: B plus the same 200 human labels")
fit_eval(tr, y_true[tr], "ceiling: trained on every human label")
print("\npairs the rules abstained on:", int((~lab).sum()), "| true matches among them:", int(y_true[~lab].sum()))

**Read it - and notice the failure in row A.** Leaving the pairs the rules abstained on out of training looks harmless, but those are exactly the *ambiguous* pairs. A model that never saw one extrapolates badly there: at the usual 0.5 threshold it flags thousands of non-matches (precision collapses to about 0.2), even though the weak labels themselves were 97%+ accurate. Abstentions are not a random sample. Row B fixes it with a prior: matches are rare, so an unlabelled pair is far more likely a non-match; treating abstentions as negatives gives an F1 close to the all-labels ceiling with **zero** human labels. Row C shows 200 random human labels are fragile (they contain only a handful of positives). Row D, weak labels for volume plus a few human labels, is the practical recipe - and the human set should stay out of training whenever you can afford it, because it is your only independent grader.

## Part 5 - Group: from pair scores to identities

Pair decisions must become **identities** (clusters of records). Two simple ways, and one common failure:

1. **Connected components** of the graph of predicted matches. Easy, but one wrong link merges two whole identities (*chain merging*).
2. **One-to-one assignment** when each record can have at most one partner in the other file: keep each record's best-scoring pair, mutually.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

proba_all = m_mix.predict_proba(X)[:, 1]
pred = proba_all >= 0.5
ids_a = {k: i for i, k in enumerate(A.index)}; ids_b = {k: i + len(A) for i, k in enumerate(B.index)}
rows = [ids_a[a] for a, k in zip(RA.rec_id[pred], pred[pred])]; cols = [ids_b[b] for b in RB.rec_id[pred]]
g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(A) + len(B),) * 2)
n_comp, labels = connected_components(g, directed=False)
sizes = pd.Series(labels).value_counts()
print("connected components: size distribution (records per identity):", {int(k): int(v) for k, v in sizes.value_counts().sort_index().items()})

# one-to-one: mutual best partner
d = pd.DataFrame({"a": RA.rec_id.values, "b": RB.rec_id.values, "s": proba_all})[pred]
best_a = d.loc[d.groupby("a").s.idxmax()]; best_b = d.loc[d.groupby("b").s.idxmax()]
mutual = best_a.merge(best_b, on=["a", "b"])
p1, r1, f1 = prf(np.ones(len(mutual), dtype=int), np.array([(a, b) in truth for a, b in zip(mutual.a, mutual.b)]))
tp_all = sum((a, b) in truth for a, b in zip(mutual.a, mutual.b))
print(f"one-to-one pairs kept: {len(mutual):,} | correct: {tp_all:,} | precision {tp_all/len(mutual):.3f} | recall {tp_all/len(truth):.3f}")

## Part 6 - Audit when you have no ground truth

In a real platform nobody hands you `true_links`. What you can do:

1. **Review a random sample of predicted matches** and report precision with a confidence interval, not a bare number. (Here a "reviewer" is the hidden truth.)
2. **Look for structural impossibilities**: clusters larger than the physical limit, one record matched to many, a cluster that mixes two states.
3. **Sweep the threshold** and see how stable the picture is; a result that collapses between 0.4 and 0.6 is not trustworthy.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (0, 1)
    p = k / n; d = 1 + z*z/n; c = p + z*z/(2*n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((c-h)/d, (c+h)/d)

idx = np.where(pred)[0]; samp = np.random.default_rng(1).choice(idx, size=200, replace=False)
k = int(y_true[samp].sum()); lo, hi = wilson(k, len(samp))
print(f"reviewed 200 predicted matches: {k} correct -> precision {k/200:.3f}, 95% interval [{lo:.3f}, {hi:.3f}]")
print("true precision on all predicted matches:", f"{y_true[idx].mean():.3f}", "(inside the interval, as it should be about 95% of the time)")

print(f"\n{'threshold':>9}{'predicted':>11}{'precision':>11}{'recall':>8}")
for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    pr = proba_all >= t; tp = (pr & y_true).sum()
    print(f"{t:>9.1f}{pr.sum():>11,}{tp/max(pr.sum(),1):>11.3f}{tp/y_true.sum():>8.3f}")

## What to carry into a real identification problem

1. **Block with several keys and measure completeness on a labelled sample** - it is the ceiling.
2. **Write labelling functions that disagree on purpose**; report each one's coverage and conflicts *before* looking at any truth.
3. **Train on weak labels for volume and keep a small human-labelled set out of training** to calibrate and grade.
4. **Group carefully**: prefer one-to-one or constrained clustering over raw connected components, and watch the cluster-size histogram.
5. **Audit with sampled review and a confidence interval**, and re-run the audit every time the upstream data changes.

*Limits of this lab:* Febrl is a synthetic benchmark with clean schemas; real browser or device signals are higher-dimensional, drift over time, and are adversarial (someone is trying to look like somebody else). The pipeline shape stays the same; the features, blocking keys and rules change.